# 0.4 Introduction to Calculus

**Goal:** understand change, accumulation, and the calculus behind loss optimization and probability densities.

**Prerequisites:** the previous three notebooks, especially functions, exponents, logarithms, and summation. **Runtime:** Python 3.10+, standard library only. Saved outputs are included; run cells in order.

## Contents
1. Limits
2. Continuity
3. Derivatives
4. Differentiation rules
5. Chain rule
6. Partial derivatives and gradients
7. Integration
8. Indefinite and definite integrals
9. AI/ML applications: gradient descent, losses, and PDFs
10. Practice and solutions

## 1. Limits

The statement $\lim_{x\to a}f(x)=L$ means that $f(x)$ can be made arbitrarily close to $L$ by taking $x$ sufficiently close to, but not necessarily equal to, $a$. More precisely: for every $\varepsilon>0$, there exists $\delta>0$ such that $0<|x-a|<\delta$ implies $|f(x)-L|<\varepsilon$ (within the domain).

The left-hand limit approaches from $x<a$; the right-hand limit from $x>a$. A finite two-sided limit exists only if both one-sided limits exist and agree.

**Worked example:**
$$\lim_{x\to1}\frac{x^2-1}{x-1}=\lim_{x\to1}(x+1)=2.$$
The original expression is undefined at 1, but has a limit there. Cancellation is valid for $x\ne1$, exactly what a limit permits.

For finite limits, sums, products, and quotients follow arithmetic rules; the quotient rule for limits requires a nonzero denominator limit. A form such as $0/0$ is **indeterminate**, not zero and not a computed answer.

For $f(x)=|x|/x$, the left-hand limit at 0 is $-1$, the right-hand limit is 1, and the two-sided limit does not exist. Infinite limits describe unbounded growth, not a finite real limit. For example, $1/x^2\to+\infty$ as $x\to0$, while $1/x\to0$ as $x\to+\infty$.

A table of nearby values provides evidence and intuition, not a proof of a limit.

In [1]:
import math
for h in [0.1, 0.01, 0.001]:
    left, right = 1 - h, 1 + h
    print(f"h={h}: left={(left**2-1)/(left-1):.6f}, right={(right**2-1)/(right-1):.6f}")
for x in [-0.001, 0.001]:
    print("|x|/x at", x, "=", abs(x)/x)

h=0.1: left=1.900000, right=2.100000
h=0.01: left=1.990000, right=2.010000
h=0.001: left=1.999000, right=2.001000
|x|/x at -0.001 = -1.0
|x|/x at 0.001 = 1.0


## 2. Continuity

A function is continuous at an interior point $a$ if:

1. $f(a)$ is defined.
2. $\lim_{x\to a}f(x)$ exists.
3. That limit equals $f(a)$.

At a domain endpoint, use the appropriate one-sided limit. Polynomials are continuous everywhere; rational functions are continuous where their denominator is nonzero. Exponentials are continuous on $\mathbb R$ and logarithms on $(0,\infty)$.

**Worked example:** define $g(x)=(x^2-1)/(x-1)$ when $x\ne1$ and $g(1)=c$. The function is continuous at 1 exactly when $c=2$. This fills a **removable discontinuity**. A step from 0 for $x<0$ to 1 for $x\ge0$ has a **jump** that cannot be fixed by changing only its value at 0.

Continuity does not imply differentiability: $|x|$ is continuous at 0 but has a corner there. Differentiability implies continuity at that point. A continuous function on a closed bounded interval attains a minimum and a maximum, a useful existence guarantee in optimization.

In [2]:
def continuous_extension(x):
    if x == 1:
        return 2.0
    return (x*x - 1) / (x - 1)

print([continuous_extension(x) for x in [0.9, 1, 1.1]])

[1.9, 2.0, 2.1]


## 3. Derivatives: instantaneous rate of change

The derivative at $a$ is
$$f'(a)=\lim_{h\to0}\frac{f(a+h)-f(a)}{h},$$
when the finite limit exists. The quotient is a secant slope; the limiting value is the tangent slope. Equivalent notation includes $df/dx$ and $\frac{d}{dx}f(x)$.

**Worked derivation:** for $f(x)=x^2$,
$$\frac{(x+h)^2-x^2}{h}=2x+h\longrightarrow2x.$$
Thus $f'(3)=6$, and a local linear approximation is $f(3+\Delta x)\approx9+6\Delta x$. For $\Delta x=0.01$, it gives 9.06, close to the exact 9.0601.

If $y$ has units meters and $x$ seconds, $dy/dx$ has units meters per second. Positive derivative suggests local increase; negative suggests local decrease.

A **stationary point** has derivative zero. It need not be a minimum: $x^3$ has derivative zero at 0 but keeps increasing. For a twice-differentiable scalar function with $f'(a)=0$, $f''(a)>0$ implies a strict local minimum and $f''(a)<0$ a strict local maximum; $f''(a)=0$ is inconclusive. Check nondifferentiable points and boundaries as well when searching for extrema.

A finite difference approximates a derivative. Central differences use $(f(x+h)-f(x-h))/(2h)$. Extremely small $h$ can amplify floating point cancellation; a finite difference is a numerical check, not an exact symbolic derivative.

In [3]:
def central_difference(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2*h)

for h in [1e-1, 1e-3, 1e-5]:
    print("h:", h, "derivative of x^2 at 3:", central_difference(lambda x: x*x, 3, h))
print("Left slope of |x| at 0:", (abs(0) - abs(-1e-5)) / 1e-5)
print("Right slope of |x| at 0:", (abs(1e-5) - abs(0)) / 1e-5)

h: 0.1 derivative of x^2 at 3: 6.000000000000005
h: 0.001 derivative of x^2 at 3: 5.999999999999339
h: 1e-05 derivative of x^2 at 3: 6.000000000039306
Left slope of |x| at 0: -1.0
Right slope of |x| at 0: 1.0


## 4. Differentiation rules

Assume functions are differentiable and expressions are in their real domains.

| Rule/function | Derivative |
|---|---|
| Constant $c$ | $0$ |
| $x^p$ | $px^{p-1}$ where defined/differentiable |
| $af(x)+bg(x)$ | $af'(x)+bg'(x)$ |
| Product $f(x)g(x)$ | $f'(x)g(x)+f(x)g'(x)$ |
| Quotient $f(x)/g(x)$ | $(f'g-fg')/g^2$, $g(x)\ne0$ |
| $e^x$ | $e^x$ |
| $a^x$, $a>0$ | $a^x\ln a$ |
| $\ln x$, $x>0$ | $1/x$ |
| $\log_a x$, $a>0,a\ne1,x>0$ | $1/(x\ln a)$ |
| $\sin x$ | $\cos x$ (angles in radians) |
| $\cos x$ | $-\sin x$ (angles in radians) |

**Worked examples:**

- $\frac d{dx}(3x^3-2x+5)=9x^2-2$.
- $\frac d{dx}(xe^x)=e^x+xe^x$.
- $\frac d{dx}\frac{x}{1+x}=\frac{(1+x)-x}{(1+x)^2}=1/(1+x)^2$ for $x\ne-1$.

The derivative of a product is not just the product of derivatives; the derivative of a quotient is not their quotient.

In [4]:
examples = [
    ("polynomial", lambda x: 3*x**3 - 2*x + 5, lambda x: 9*x*x - 2),
    ("product", lambda x: x*math.exp(x), lambda x: (1+x)*math.exp(x)),
    ("quotient", lambda x: x/(1+x), lambda x: 1/(1+x)**2),
]
for name, f, derivative in examples:
    numerical = central_difference(f, 0.5)
    analytical = derivative(0.5)
    print(name, "analytic:", analytical, "numerical:", numerical)
    assert math.isclose(numerical, analytical, rel_tol=1e-7, abs_tol=1e-7)

polynomial analytic: 0.25 numerical: 0.25000000034580694
product analytic: 2.4730819060501923 numerical: 2.473081906140262
quotient analytic: 0.4444444444444444 numerical: 0.4444444444640094


## 5. Chain rule

For a composition $y=f(g(x))$,
$$\frac{dy}{dx}=f'(g(x))g'(x).$$

Differentiate the outer function at the inner value, then multiply by the inner derivative.

**Worked examples:**

- $(3x+1)^4$: outer derivative $4(3x+1)^3$, inner derivative 3; result $12(3x+1)^3$.
- $\ln(1+x^2)$: derivative $2x/(1+x^2)$.
- $e^{-2x}$: derivative $-2e^{-2x}$.

For the sigmoid $s(z)=1/(1+e^{-z})$, differentiating gives
$$s'(z)=\frac{e^{-z}}{(1+e^{-z})^2}=s(z)(1-s(z)).$$

**Backpropagation connection:** if $z=wx+b$, $\hat y=s(z)$, and $L=(\hat y-y)^2$, then
$$\frac{\partial L}{\partial w}=2(\hat y-y)s(z)(1-s(z))x.$$
This multiplies the loss-to-prediction, prediction-to-score, and score-to-weight derivatives. Omitting any factor changes the gradient.

In [5]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    exp_z = math.exp(z)
    return exp_z / (1 + exp_z)

x, y, w, b = 2.0, 1.0, 0.3, -0.1
prediction = sigmoid(w*x + b)
analytic = 2*(prediction-y)*prediction*(1-prediction)*x
numeric = central_difference(lambda weight: (sigmoid(weight*x+b)-y)**2, w)
print("Prediction:", prediction)
print("Weight gradient:", analytic, "finite difference:", numeric)
assert math.isclose(analytic, numeric, rel_tol=1e-7)

Prediction: 0.6224593312018546
Weight gradient: -0.35489383469854746 finite difference: -0.3548938347003316


## 6. Partial derivatives and gradients

For a multivariable function $f(w,b)$, a partial derivative changes one variable while holding the other fixed.

**Worked example:** $f(w,b)=w^2+3wb+b^2$ gives
$$\frac{\partial f}{\partial w}=2w+3b,\qquad
\frac{\partial f}{\partial b}=3w+2b.$$

The **gradient** collects partial derivatives:
$$\nabla f=\begin{bmatrix}\partial f/\partial w\\\partial f/\partial b\end{bmatrix}.$$
For a differentiable function, its gradient points in the direction of steepest local increase under the Euclidean norm; the negative gradient gives steepest local decrease. It is a local statement, not a guarantee of finding a global minimum.

For linear predictions $\hat y_i=wx_i+b$ and $n$ paired examples,
$$L(w,b)=\frac1n\sum_i(wx_i+b-y_i)^2,$$
$$\frac{\partial L}{\partial w}=\frac2n\sum_i(wx_i+b-y_i)x_i,\qquad
\frac{\partial L}{\partial b}=\frac2n\sum_i(wx_i+b-y_i).$$

A convention with $1/(2n)$ in the loss removes the factor 2 from both gradients. Always differentiate the actual loss definition.

In [6]:
w, b = 2.0, -1.0
f = lambda w, b: w*w + 3*w*b + b*b
analytic_gradient = (2*w + 3*b, 3*w + 2*b)
numeric_gradient = (
    central_difference(lambda value: f(value, b), w),
    central_difference(lambda value: f(w, value), b),
)
print("Analytic gradient:", analytic_gradient)
print("Numerical gradient:", numeric_gradient)
for exact, approximate in zip(analytic_gradient, numeric_gradient):
    assert math.isclose(exact, approximate, abs_tol=1e-7)

Analytic gradient: (1.0, 4.0)
Numerical gradient: (0.9999999999621422, 3.9999999999928977)


## 7. Introduction to integration

Integration describes accumulation and signed area, and it reverses differentiation through antiderivatives.

Partition $[a,b]$ into narrow widths $\Delta x$. A sum $\sum_i f(x_i^*)\Delta x$ approximates accumulated area. For a continuous function, the limit as the partition becomes fine is the definite integral. Positive graph values contribute positive area; negative values contribute negative area. Geometric area may require integrating $|f(x)|$ instead.

If $f(x)$ is a rate in units per second and $x$ is time in seconds, its integral has units of the accumulated quantity.

## 8. Indefinite and definite integrals

An **indefinite integral** is a family of antiderivatives:
$$\int f(x)\,dx=F(x)+C,\qquad F'(x)=f(x).$$
The constant $C$ is essential because constants differentiate to zero.

| Integrand | Antiderivative |
|---|---|
| $x^p$, $p\ne-1$ | $x^{p+1}/(p+1)+C$, on a suitable real domain |
| $1/x$ | $\ln|x|+C$ on an interval excluding 0 |
| $e^x$ | $e^x+C$ |
| $a f(x)+b g(x)$ | $aF(x)+bG(x)+C$ |

**Substitution** reverses the chain rule: letting $u=1+x^2$, $du=2x\,dx$ gives
$$\int2xe^{1+x^2}\,dx=e^{1+x^2}+C.$$

A **definite integral** produces a number. The Fundamental Theorem of Calculus states, for continuous $f$ and an antiderivative $F$ on $[a,b]$:
$$\int_a^b f(x)\,dx=F(b)-F(a).$$
It also connects accumulation to differentiation: $\frac{d}{dx}\int_a^x f(t)\,dt=f(x)$ for continuous $f$.

**Worked examples:**
$$\int(3x^2+2x)\,dx=x^3+x^2+C,$$
$$\int_0^2(3x^2+2x)\,dx=[x^3+x^2]_0^2=12.$$

Useful rules: $\int_a^a f=0$, $\int_a^b f=-\int_b^a f$, and $\int_a^c f+\int_c^b f=\int_a^b f$. For example $\int_{-1}^{1}x\,dx=0$ despite nonzero geometric area.

Numerical integration approximates a definite integral when an antiderivative is inconvenient. The trapezoidal rule adds trapezoid areas; refine the grid and check convergence rather than assuming an exact answer.

In [7]:
def trapezoid(f, a, b, intervals=1000):
    if not isinstance(intervals, int) or intervals <= 0:
        raise ValueError("intervals must be a positive integer")
    width = (b-a) / intervals
    interior = sum(f(a + i*width) for i in range(1, intervals))
    return width * (0.5*f(a) + interior + 0.5*f(b))

for intervals in [10, 100, 1000]:
    area = trapezoid(lambda x: 3*x*x + 2*x, 0, 2, intervals)
    print("Intervals:", intervals, "estimate:", area, "absolute error:", abs(area-12))
assert math.isclose(trapezoid(lambda x: 3*x*x + 2*x, 0, 2), 12, abs_tol=5e-6)
print("Signed integral of x:", trapezoid(lambda x: x, -1, 1))

Intervals: 10 estimate: 12.040000000000003 absolute error: 0.0400000000000027
Intervals: 100 estimate: 12.000399999999999 absolute error: 0.00039999999999906777
Intervals: 1000 estimate: 12.000004 absolute error: 4.000000000559112e-06
Signed integral of x: 4.1744385725905886e-17


## 9. AI/ML applications

### A. Gradient descent on a loss function

For parameters $\theta$ and learning rate $\eta>0$:
$$\theta_{t+1}=\theta_t-\eta\nabla L(\theta_t).$$

Compute every component of the gradient at the **same current parameter values** before updating. For smooth convex quadratics, suitable step sizes converge to a global minimum; excessively large steps can diverge. Nonconvex objectives may have local minima or saddle points, and stopping at a small gradient does not by itself prove global optimality.

For $L(w)=(w-3)^2$, the derivative is $2(w-3)$ and the error recurrence is
$$w_{t+1}-3=(1-2\eta)(w_t-3).$$
Convergence from a general starting value requires $|1-2\eta|<1$, or $0<\eta<1$. At $\eta=1$ errors oscillate; at $\eta>1$ they grow unless already at the optimum.

In [8]:
for learning_rate in [0.1, 1.0, 1.1]:
    w = 0.0
    losses = [(w-3)**2]
    for _ in range(12):
        w -= learning_rate * 2*(w-3)
        losses.append((w-3)**2)
    print(f"rate={learning_rate}: final w={w:.6f}, initial loss={losses[0]:.6f}, final loss={losses[-1]:.6f}")

rate=0.1: final w=2.793842, initial loss=9.000000, final loss=0.042501
rate=1.0: final w=0.000000, initial loss=9.000000, final loss=9.000000
rate=1.1: final w=-23.748301, initial loss=9.000000, final loss=715.471625


### B. Fit a line with partial derivatives

Use the MSE gradients derived above to fit $y=2x+1$ on a tiny deterministic dataset. Centered input values make this demonstration easy to optimize. Real training requires separate validation/test data and does not usually achieve zero prediction error.

In [9]:
xs, ys = [-2, -1, 0, 1, 2], [-3, -1, 1, 3, 5]

def mse(w, b):
    return sum((w*x+b-y)**2 for x, y in zip(xs, ys)) / len(xs)

def mse_gradient(w, b):
    residuals = [w*x+b-y for x, y in zip(xs, ys)]
    n = len(xs)
    return (2*sum(r*x for r, x in zip(residuals, xs))/n,
            2*sum(residuals)/n)

w, b = 0.4, -0.3
exact_w, exact_b = mse_gradient(w, b)
assert math.isclose(exact_w, central_difference(lambda v: mse(v, b), w), rel_tol=1e-7)
assert math.isclose(exact_b, central_difference(lambda v: mse(w, v), b), rel_tol=1e-7)
w, b = 0.0, 0.0
initial_loss = mse(w, b)
for step in range(100):
    dw, db = mse_gradient(w, b)
    w, b = w - 0.1*dw, b - 0.1*db
print("Initial MSE:", initial_loss)
print(f"Fitted slope={w:.8f}, intercept={b:.8f}, MSE={mse(w,b):.3e}")
assert math.isclose(w, 2, abs_tol=1e-7)
assert math.isclose(b, 1, abs_tol=1e-7)

Initial MSE: 9.0
Fitted slope=2.00000000, intercept=1.00000000, MSE=4.150e-20


### C. Probability density functions (PDFs)

A probability density $f$ must satisfy $f(x)\ge0$ and $\int_{-\infty}^{\infty}f(x)\,dx=1$. For a continuous random variable,
$$P(a\le X\le b)=\int_a^b f(x)\,dx.$$

A density value is not a point probability and can exceed 1; for a variable with a density, $P(X=x)=0$. Its CDF is $F(x)=\int_{-\infty}^x f(t)\,dt$. Where the density is continuous, $F'(x)=f(x)$.

**Worked example:** let $f(x)=2x$ on $[0,1]$, and 0 elsewhere. It is nonnegative and $\int_0^1 2x\,dx=[x^2]_0^1=1$. Its CDF is 0 below 0, $x^2$ within $[0,1]$, and 1 above 1.

$$P(0.2\le X\le0.7)=0.7^2-0.2^2=0.45.$$

Although $f(0.8)=1.6>1$, total integrated probability remains 1. Endpoints do not affect these interval probabilities. Unbounded-support PDFs require **improper integrals**, defined through limits; for example, $\int_0^\infty e^{-x}\,dx=\lim_{b\to\infty}(1-e^{-b})=1$.

In [10]:
def density(x):
    return 2*x if 0 <= x <= 1 else 0.0

def cdf(x):
    if x < 0:
        return 0.0
    if x > 1:
        return 1.0
    return x*x

normalization = trapezoid(density, 0, 1)
probability = trapezoid(density, 0.2, 0.7)
print("Total probability:", normalization)
print("Interval probability:", probability)
print("CDF difference:", cdf(0.7)-cdf(0.2))
print("Density at 0.8:", density(0.8))
assert math.isclose(normalization, 1, abs_tol=1e-12)
assert math.isclose(probability, 0.45, abs_tol=1e-12)
assert math.isclose(central_difference(cdf, 0.5), density(0.5), rel_tol=1e-7)

Total probability: 1.0
Interval probability: 0.4499999999999999
CDF difference: 0.44999999999999996
Density at 0.8: 1.6


## Practice

1. Find $\lim_{x\to2}(x^2-4)/(x-2)$ and explain why direct substitution fails.
2. Choose $c$ to make $f(x)=\sin x/x$ for $x\ne0$, $f(0)=c$, continuous. Use the standard limit $\lim_{x\to0}\sin x/x=1$ (radians).
3. Differentiate $4x^3-5x^2+7$, $x\ln x$ for $x>0$, and $\ln(2x+1)$.
4. For $L(w)=(2w-5)^2$, find $L'(w)$ and the minimizer.
5. Find both partial derivatives of $f(w,b)=w^2+wb+3b^2$.
6. Starting at $w=0$, take one gradient descent step on $L(w)=(w-4)^2$ with $\eta=0.1$.
7. Compute $\int(6x^2-4)\,dx$ and $\int_0^2(6x^2-4)\,dx$.
8. Normalize $f(x)=cx^2$ on $[0,1]$, zero elsewhere; calculate $P(X\le0.5)$.
9. Is $f'(a)=0$ sufficient for a minimum? Give a counterexample.
10. For $L(w,b)=(wx+b-y)^2$ with a single observation, derive both partial derivatives.

## Solutions

1. Factor $(x-2)(x+2)$ and cancel for $x\ne2$; limit 4. Substitution in the original gives the indeterminate form $0/0$.
2. $c=1$, so the function value matches the given limit.
3. $12x^2-10x$; product rule gives $\ln x+1$; chain rule gives $2/(2x+1)$ for $x>-1/2$.
4. $L'(w)=2(2w-5)\times2=4(2w-5)$; $w=2.5$. The loss is nonnegative and zero there; equivalently $L''=8>0$.
5. $\partial f/\partial w=2w+b$; $\partial f/\partial b=w+6b$.
6. Gradient at 0 is $-8$; new weight $0-0.1(-8)=0.8$. Loss falls from 16 to $3.2^2=10.24$.
7. $2x^3-4x+C$; definite value $[2x^3-4x]_0^2=16-8=8$.
8. $\int_0^1cx^2\,dx=c/3=1$, so $c=3$. Probability $=\int_0^{0.5}3x^2\,dx=(0.5)^3=0.125$.
9. No. $f(x)=x^3$ has $f'(0)=0$ but neither a local minimum nor maximum there. Also, a minimum need not have a derivative: $|x|$ has a minimum at 0.
10. Let $r=wx+b-y$. The chain rule gives $\partial L/\partial w=2rx$ and $\partial L/\partial b=2r$.

## Formula recap

- Limit: behavior near an input; continuity: that behavior matches the function value.
- Derivative: instantaneous slope; partial derivative: slope holding other coordinates fixed.
- Chain rule: multiply derivatives along a composition.
- Gradient descent: subtract a scaled gradient, using all components evaluated before the update.
- Indefinite integral: antiderivative plus a constant; definite integral: signed accumulation.
- PDF interval probability: area under a nonnegative normalized density.

**Readiness check:** derive the MSE gradient, verify it with a finite difference, explain learning-rate divergence, and integrate a simple PDF before moving on to statistics.

**Next syllabus topic:** Module 1 — Introduction to Statistics and Data, in the repository [main syllabus](../README.md).